# reconciliation.ipynb — сверка Google Sheets с месячным Excel CRM

Находит, каких звонков не хватает в Google Sheets по сравнению
с полным месячным Excel CRM, и во сколько минут это обходится.
Ячейка 6 умеет дописать пропавшие звонки обратно в нужные вкладки.

Нужен, потому что дневной Excel CRM теряет первую запись периода:
при дневных файлах это тридцать звонков за месяц, при месячном —
один. Значит месячный файл полнее суммы дневных.

Порядок работы:
    1. Ячейка 1 — настройки
    2. Ячейка 2 — чтение месячного Excel CRM
    3. Ячейка 3 — чтение вкладок Google Sheets
    4. Ячейка 4 — сверка и отчёт
    5. Ячейка 5 — сверка CRM и АТС по исходу звонка
    6. Ячейка 6 — дозапись пропавших звонков (по умолчанию вхолостую)

Порядок не случайный: сперва оба разбора — чего не хватает и где
исход разошёлся с АТС, — и только потом запись. Так решение
о дозаписи принимается, когда видна вся картина.

## ЯЧЕЙКА 1 — НАСТРОЙКИ

In [ ]:
import calendar
import re
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

from etl_core import (
    DATA_DIR, CREDENTIALS, env, DEMO,
    DateResolver, SheetsLoader, RU_MONTHS, month_title, preview, log,
    bill_minutes,
)

BASE_DIR = str(DATA_DIR)   # выгрузки лежат в папке data/ в корне репозитория

# Сверка сравнивает минуты CRM с посчитанной формулой округления на
# дневных листах (колонка G). В демо-режиме формулы в Excel лежат текстом
# и не считаются — сверка нашла бы ошибки там, где их нет.
if DEMO:
    raise RuntimeError("Демо-режим: сверка не запускается — ей нужны "
                       "посчитанные формулы Google Sheets.")
# CREDENTIALS импортируется из etl_core: credentials.json в корне, в git не попадает
# Спрашивать проект и год-месяц при запуске. False — молча взять
# значения ниже, без вопросов.
ASK = True

# Проекты. Смотри PROJECTS ниже — там же добавляются новые.
# Структура Excel CRM и сама логика сверки от проекта не зависят,
# отличаются только эти три настройки.
#
# Omega убран — проект закрыт.
#
# Проекты Мониторинга (Beta, Gamma, Delta, Epsilon, Zeta,
# Eta) пока не добавлены: их дневные вкладки собирает
# monitoring.ipynb по другой раскладке — из CRM туда идут только исходящие
# звонки, входящие берутся из почасового отчёта АТС, а не из
# реестра. Точные номера колонок под эту раскладку не известны,
# добавлять их сюда вслепую значит рисковать неверными выводами.
#
# {month} в имени файла подставляется из года-месяца ниже — Alpha
# за август ищет файл Alpha_Август.xlsx, за июль — Alpha_Июль.xlsx.
PROJECTS = {
    "alpha": {
        "month_file_tpl": "Alpha_{month}.xlsx",
        "spreadsheet_id": env("ALPHA_SHEET_ID"),
        "spreadsheet_title": "Alpha {month} {year}",
        # Заказчик не тарифицирует очень короткие звонки: 5 секунд и
        # короче — 0 минут, а не округление вверх до одной минуты.
        # У кого такого правила нет, просто не пишет эту строку —
        # тогда действует обычное округление вверх.
        "free_under_sec": 5,
    },
}

# Значения по умолчанию. Их же скрипт предлагает в вопросах —
# достаточно нажать Enter.
PROJECT = "alpha"
YEAR, MONTH = 2026, 8


def ask_project(default: str) -> str:
    """
    Спрашивает проект. Если он в PROJECTS всего один, вопрос не
    нужен — молча берём default, отвечать не на что.
    """
    names = list(PROJECTS)
    if len(names) < 2:
        return default
    while True:
        raw = input(f"Проект [{', '.join(names)}] "
                    f"[{default}]: ").strip().lower()
        if not raw:
            return default
        if raw in PROJECTS:
            return raw
        print(f"   Не знаю такой. Есть: {', '.join(names)}")


def ask_month(default_year: int, default_month: int) -> tuple:
    """
    Спрашивает год и месяц. Понимает «2026-08» и просто «08» — тогда
    год берётся по умолчанию. Кривой ввод переспрашиваем: сверить не
    тот месяц значит впустую прогнать всю ячейку.
    """
    while True:
        raw = input(f"Год-месяц (ГГГГ-ММ) "
                    f"[{default_year}-{default_month:02d}]: ").strip()
        if not raw:
            return default_year, default_month
        try:
            if "-" in raw:
                y, m = raw.split("-")
                y, m = int(y), int(m)
            else:
                y, m = default_year, int(raw)
            if 1 <= m <= 12:
                return y, m
        except ValueError:
            pass
        print("   Не понял. Формат: 2026-08 или просто 08")


if ASK:
    PROJECT = ask_project(PROJECT)
    YEAR, MONTH = ask_month(YEAR, MONTH)

_cfg = PROJECTS[PROJECT]
MONTH_FILE = _cfg["month_file_tpl"].format(month=RU_MONTHS[MONTH])
MONTH_SHEET = None          # None — первый лист файла

SPREADSHEET_ID = _cfg["spreadsheet_id"]
SPREADSHEET_TITLE = _cfg["spreadsheet_title"]
USE_TITLE = True

# Раскладка колонок на дневной вкладке:
# A ID | B Направление | C Телефон | D Начало | E Конец | F Сек | G Минуты
COL_ID, COL_PHONE, COL_SEC, COL_MIN = 0, 2, 5, 6

# Правило округления минут для тарификации этого проекта.
FREE_UNDER_SEC = _cfg.get("free_under_sec")


def calc_minutes(sec):
    """
    Секунды в минуты для тарификации — с учётом правила заказчика.

    Обычно округление ВВЕРХ: разговор в 61 секунду — уже две минуты.
    Но если для проекта задан FREE_UNDER_SEC (у Alpha — 5 сек),
    звонки короче или равные этому порогу считаются за 0 минут, а не
    округляются до одной. Без этого правила сверка находила бы
    «ошибку формулы» в каждой такой строке — хотя лист посчитал
    её ровно так, как договорился заказчик.

    Принимает Series — используется и на всём Excel CRM разом
    (ячейка 2), и на одном дне при проверке листа (ячейка 4).

    Само правило — в etl_core.bill_minutes: там его проверяют тесты.
    """
    return bill_minutes(sec, FREE_UNDER_SEC)


def minutes_formula(row_no: int) -> str:
    """
    Формула колонки «Округление, минуты» для новой строки.

    Пишется как формула, а не готовым числом — тогда дозаписанная
    ячейка 6 строка ведёт себя как соседние, а не отличается от них.
    Если для проекта задан FREE_UNDER_SEC, формула повторяет то же
    правило заказчика, что применяет calc_minutes() к остальному Excel CRM —
    иначе новая строка округлялась бы иначе, чем весь остальной день.
    """
    cell = f"{chr(65 + COL_SEC)}{row_no}"
    if FREE_UNDER_SEC:
        return f"=IF({cell}<={FREE_UNDER_SEC},0,CEILING({cell}/60,1))"
    return f"=CEILING({cell}/60,1)"

# Дозапись в ячейке 6. True — только показать, ничего не менять.
DRY_RUN = True

## ЯЧЕЙКА 2 — МЕСЯЧНАЯ ВЫГРУЗКА

In [ ]:
def norm_phone(v) -> str:
    """
    Приводит номер к одному виду: только цифры.

    Из Excel номер приезжает числом с плавающей точкой — str() даёт
    '901234567.0'. Из Google Sheets тот же номер приходит строкой
    '901234567'. Без нормализации ключи не совпадут НИ РАЗУ, и сверка
    объявит потерянными все звонки месяца.
    """
    if v is None:
        return ""
    s = str(v).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return re.sub(r"\D", "", s)


def call_key(cid, phone) -> tuple:
    """
    Ключ строки: ID звонка плюс номер телефона.

    Одного ID мало. При переводе звонка между операторами CRM отдаёт
    ДВЕ строки с одним ID, но разными номерами — это две ноги одного
    разговора. По одному ID они схлопываются в множестве, и сверка
    показывает потерю, которой нет. На реальных данных такие пары были.

    Полные дубли (все поля совпадают) этим ключом схлопываются
    намеренно: это одна и та же строка, вставленная дважды.
    Их считает отдельная проверка в ячейке 4.
    """
    return (int(cid), norm_phone(phone))


def read_month_export() -> pd.DataFrame:
    """
    Читает месячный Excel CRM.

    В конце файла обычно стоит строка «Итого» без ID — её отсекаем,
    иначе она попадёт в данные и удвоит сумму минут.

    Дни ЧУЖИХ месяцев тоже отсекаем. Это важно: CRM теряет первую
    запись запрошенного периода, поэтому месяц правильно качать с
    буфером — с последнего числа предыдущего месяца, чтобы потерялась
    запись буферного дня, а не первое число. Но тогда в файле лежит
    день, которого нет среди вкладок, и без фильтра ВСЕ его звонки
    (у крупного проекта это больше тысячи) попадали бы в «нет в Google Sheets»
    как фантомные пропуски.
    """
    p = Path(BASE_DIR) / MONTH_FILE
    if not p.exists():
        raise FileNotFoundError(
            f"Файл не найден: {p}\n"
            f"Положите месячный файл Excel CRM в {BASE_DIR}"
        )

    df = pd.read_excel(p, sheet_name=MONTH_SHEET or 0)
    df.columns = [str(c).strip() for c in df.columns]

    need = ["ID звонка", "Номер телефона", "Время начала звонка",
            "Продолжительность, сек"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise KeyError(f"В файле нет колонок {missing}. "
                       f"Есть: {list(df.columns)}")

    before = len(df)
    df = df[df["ID звонка"].notna()].copy()
    if len(df) < before:
        log.info("Отсечено служебных строк без ID: %d", before - len(df))

    df["_id"] = pd.to_numeric(df["ID звонка"], errors="coerce").astype("Int64")
    df["_phone"] = df["Номер телефона"].map(norm_phone)
    df["_key"] = [call_key(i, p) for i, p in zip(df["_id"], df["_phone"])]

    dt = pd.to_datetime(df["Время начала звонка"], errors="coerce")
    df["_dt"] = dt
    df["_d"] = dt.dt.date
    sec = pd.to_numeric(df["Продолжительность, сек"], errors="coerce").fillna(0)
    df["_sec"] = sec
    df["_min"] = calc_minutes(sec)

    df = df.dropna(subset=["_d"])

    # Оставляем только сверяемый месяц. Что отсеяли — показываем:
    # буферный день это норма, а вот чужой месяц целиком значит,
    # что в настройках стоит не тот файл или не тот период.
    before_filter = len(df)
    other = df[~df["_d"].map(lambda x: x.year == YEAR and x.month == MONTH)]
    if len(other):
        by_day = other.groupby("_d").size().sort_index()
        log.info("Вне сверяемого месяца отсечено строк: %d (%s)",
                 len(other),
                 ", ".join(f"{d:%d.%m}: {n}" for d, n in by_day.items()))
        df = df[df["_d"].map(lambda x: x.year == YEAR and x.month == MONTH)]

    if df.empty:
        raise ValueError(
            f"В файле {MONTH_FILE} нет ни одного дня за "
            f"{RU_MONTHS[MONTH]} {YEAR}. Проверьте, тот ли файл и тот "
            f"ли месяц выбран при запуске."
        )

    log.info("Excel CRM: %d звонков, дней %d",
             len(df), df["_d"].nunique())
    return df


month = read_month_export()

print(f"\nПериод в файле: {month['_dt'].min()} — {month['_dt'].max()}")
print(f"Звонков: {len(month)}")
print(f"Минут (сумма округлений вверх): {int(month['_min'].sum())}")
dupes_month = len(month) - month["_key"].nunique()
if dupes_month:
    print(f"ВНИМАНИЕ: в самом Excel CRM {dupes_month} полностью "
          f"повторяющихся строк")

## ЯЧЕЙКА 3 — ВКЛАДКИ GOOGLE SHEETS

In [ ]:
def read_sheet_days(year: int, mon: int) -> tuple:
    """
    Читает дневные вкладки Google Sheets за указанный месяц.

    Возвращает ({дата: DataFrame}, {дата: имя листа}, {дата: строк в листе}).
    Последнее нужно ячейке 6: чтобы дописать строки, надо знать,
    с какой строки лист свободен.

    Минуты берутся из колонки G — то есть ровно то, что показывает
    лист, вместе с возможными ошибками формулы.
    """
    loader = SheetsLoader(CREDENTIALS, SPREADSHEET_ID)
    if USE_TITLE:
        loader.connect(title=SPREADSHEET_TITLE.format(
            month=RU_MONTHS[mon], year=year))
    else:
        loader.connect()

    d_from = date(year, mon, 1)
    d_to = date(year, mon, calendar.monthrange(year, mon)[1])
    sheets = loader.list_date_sheets(d_from, d_to)
    if not sheets:
        raise ValueError(f"В Google Sheets нет вкладок за {mon:02d}.{year}")

    q = loader.quote_sheet
    ranges = [f"{q(t)}!A2:G2000" for t in sheets.values()]
    grids = []
    for i in range(0, len(ranges), 25):
        grids += loader.batch_get(ranges[i:i + 25])

    out, last_row = {}, {}
    for d, grid in zip(sheets.keys(), grids):
        rows = []
        for r in (grid or []):
            if not r or not str(r[COL_ID]).strip():
                continue
            cid = pd.to_numeric(str(r[COL_ID]).strip(), errors="coerce")
            if pd.isna(cid):
                continue

            def cell(i):
                return str(r[i]).replace(",", ".").strip() if len(r) > i else ""

            sec = pd.to_numeric(cell(COL_SEC), errors="coerce")
            mn = pd.to_numeric(cell(COL_MIN), errors="coerce")
            rows.append({"_id": int(cid),
                         "_phone": norm_phone(cell(COL_PHONE)),
                         "_key": call_key(cid, cell(COL_PHONE)),
                         "_sec": 0.0 if pd.isna(sec) else float(sec),
                         "_min_sheet": 0.0 if pd.isna(mn) else float(mn)})
        out[d] = pd.DataFrame(rows)
        # Первая строка листа — заголовок, читали с A2. Значит первая
        # свободная строка = 1 (заголовок) + сколько вернул API + 1.
        last_row[d] = 1 + len(grid or [])
        log.info("Вкладка %s: %d звонков", sheets[d], len(rows))

    return out, sheets, last_row, loader


sheet_days, sheet_names, sheet_last_row, loader = read_sheet_days(YEAR, MONTH)

total_sheet = sum(len(v) for v in sheet_days.values())
total_min_sheet = sum(v["_min_sheet"].sum() for v in sheet_days.values() if len(v))
print(f"\nВкладок прочитано: {len(sheet_days)}")
print(f"Звонков в Google Sheets: {total_sheet}")
print(f"Минут по колонке G: {int(total_min_sheet)}")

## ЯЧЕЙКА 4 — СВЕРКА

In [ ]:
def compare() -> tuple:
    """
    Сравнивает состав звонков по дням.

    Считает четыре вещи:
      - каких строк нет в Google Sheets, но есть в Excel CRM (потеря данных),
      - каких нет в Excel CRM, но есть в Google Sheets (лишние),
      - есть ли внутри листа полностью повторяющиеся строки,
      - расходится ли колонка G с округлением вверх (ошибка формулы).
    """
    rows, lost_all, extra_all, bad_formula, dupes = [], [], [], [], []

    for d in sorted(set(month["_d"]) | set(sheet_days)):
        m = month[month["_d"] == d]
        s = sheet_days.get(d, pd.DataFrame(
            columns=["_id", "_phone", "_key", "_sec", "_min_sheet"]))

        keys_m = set(m["_key"])
        keys_s = set(s["_key"]) if len(s) else set()

        lost = keys_m - keys_s        # есть в Excel CRM, нет в Google Sheets
        extra = keys_s - keys_m       # есть в Google Sheets, нет в Excel CRM

        if len(s):
            n_dup = len(s) - len(set(s["_key"]))
            if n_dup:
                dupes.append((d, n_dup))

            calc = calc_minutes(s["_sec"])
            wrong = s[(calc - s["_min_sheet"]).abs() > 0.001]
            if len(wrong):
                bad_formula.append((d, len(wrong)))

        rows.append({
            "Дата": d.strftime("%d.%m"),
            "В Excel CRM": len(m),
            "В Google Sheets": len(s),
            "Нет у нас": len(lost),
            "Лишних": len(extra),
            "Мин Excel CRM": int(m["_min"].sum()),
            "Мин Google Sheets": int(s["_min_sheet"].sum()) if len(s) else 0,
            "Разница мин": int(m["_min"].sum()) - (int(s["_min_sheet"].sum())
                                                   if len(s) else 0),
        })

        for idx, r in m[m["_key"].isin(lost)].sort_values("_dt").iterrows():
            lost_all.append({
                "Дата": d.strftime("%d.%m"),
                "ID звонка": int(r["_id"]),
                "Время": str(r["_dt"])[11:],
                "Телефон": r["_phone"],
                "Сек": int(r["_sec"]),
                "Минут потеряно": int(r["_min"]),
                "Оператор": r.get("Оператор", ""),
                "_d": d,
                "_row": idx,          # ссылка на строку Excel CRM для ячейки 6
            })
        for k in sorted(extra):
            extra_all.append({"Дата": d.strftime("%d.%m"), "_d": d,
                              "ID звонка": k[0], "Телефон": k[1]})

    return (pd.DataFrame(rows), pd.DataFrame(lost_all),
            pd.DataFrame(extra_all), bad_formula, dupes)


by_day, lost_calls, extra_calls, bad_formula, dupes = compare()

preview(by_day, "СВЕРКА ПО ДНЯМ")

print("\nИТОГО (без разбора причин)")
print(f"  В Excel CRM     : {by_day['В Excel CRM'].sum():>6} звонков, "
      f"{by_day['Мин Excel CRM'].sum():>6} минут")
print(f"  В Google Sheets : {by_day['В Google Sheets'].sum():>6} звонков, "
      f"{by_day['Мин Google Sheets'].sum():>6} минут")


# ============================================================
#  РАЗБОР: ГДЕ ЧЬЯ ОШИБКА
#
# Сравнение по ключу (id, телефон) строгое и честно ловит настоящие
# пропуски — но у него есть слепое пятно. Если один и тот же звонок
# (один call_id) в Excel CRM и в Google Sheets записан с РАЗНЫМ телефоном —
# например, каждый источник захватил свою "ногу" переведённого
# звонка, — по ключу он не совпадёт ни разу и попадёт СРАЗУ В ОБА
# списка: как «потерянный» (ключ Excel CRM не нашёлся в Google Sheets) и
# как «лишний» (ключ Google Sheets не нашёлся в Excel CRM). Это не пропажа
# данных, это один звонок под двумя разными ключами.
#
# Раз id общий — такие пары легко найти и убрать из обоих списков.
# Это важно не только для точности отчёта: если их не убрать, ячейка
# 6 попытается дозаписать в Google Sheets звонок, который там уже есть, —
# и создаст дубль.
# ============================================================
overlap_ids = (set(lost_calls["ID звонка"]) & set(extra_calls["ID звонка"])
              if len(lost_calls) and len(extra_calls) else set())

same_call = pd.DataFrame()
if overlap_ids:
    l_g = (lost_calls[lost_calls["ID звонка"].isin(overlap_ids)]
           .groupby("ID звонка").agg(
               Дата=("Дата", "first"),
               **{"Телефон в Excel CRM": (
                   "Телефон", lambda s: ", ".join(sorted(set(s))))}))
    e_g = (extra_calls[extra_calls["ID звонка"].isin(overlap_ids)]
           .groupby("ID звонка").agg(**{"Телефон в Google Sheets": (
               "Телефон", lambda s: ", ".join(sorted(set(s))))}))
    same_call = l_g.join(e_g, how="outer").reset_index()

    lost_calls = lost_calls[~lost_calls["ID звонка"].isin(overlap_ids)].copy()
    extra_calls = extra_calls[~extra_calls["ID звонка"].isin(overlap_ids)].copy()

# ------------------------------------------------------------
# Для оставшихся «лишних» (есть в Google Sheets, нет в Excel CRM) проверяем
# гипотезу: CRM теряет самую первую запись запрошенного периода.
# Отметки времени у строк Google Sheets нет — колонка D не читается, —
# поэтому берём ID звонка как замену: в этой CRM он растёт вместе
# со временем звонка, это видно по любому Excel CRM "Все звонки",
# отсортированной по времени.
#
# Если «лишний» звонок — это САМЫЙ МАЛЕНЬКИЙ id за свой день среди
# того, что есть в Google Sheets, — он и есть первый звонок дня, и его
# отсутствие в Excel CRM объясняется этим багом, а не ошибкой у вас.
# Если нет — версия про баг ничего не объясняет, это отдельный
# случай, который стоит проверить руками.
# ------------------------------------------------------------
extra_explained, extra_unclear = pd.DataFrame(), pd.DataFrame()
if len(extra_calls):
    min_id_by_day = {d: g["_id"].min() for d, g in sheet_days.items() if len(g)}
    is_min = extra_calls.apply(
        lambda r: min_id_by_day.get(r["_d"]) == r["ID звонка"], axis=1)
    extra_explained = extra_calls[is_min].drop(columns=["_d"])
    extra_unclear = extra_calls[~is_min].drop(columns=["_d"])

print("\n" + "=" * 60)
print("ЧЬЯ ЭТО ОШИБКА")
print("=" * 60)
diff = by_day["В Google Sheets"].sum() - by_day["В Excel CRM"].sum()
print(f"\nРазница по числу звонков: {diff:+d} "
      f"(в Google Sheets минус в Excel CRM)")

if len(same_call):
    print(f"\n1. ОДИН ЗВОНОК ПОД РАЗНЫМ НОМЕРОМ ({len(same_call)}) — "
          f"не ошибка ни у кого, звонок уже есть в обоих местах, "
          f"просто с разным телефоном на ноге перевода:")
    print(same_call.to_string(index=False))

if len(extra_explained):
    print(f"\n2. ОШИБКА В EXCEL CRM ({len(extra_explained)}) — "
          f"это первый звонок дня — такая запись пропадает из Excel CRM "
          f"при выгрузке периода. У вас в Google Sheets он есть и "
          f"остаётся верным, "
          f"трогать не нужно:")
    print(extra_explained.to_string(index=False))

if len(extra_unclear):
    print(f"\n3. ЕСТЬ В GOOGLE SHEETS, НЕТ В EXCEL CRM, ПРИЧИНА НЕ ЯСНА "
          f"({len(extra_unclear)}) — это не первый звонок дня, версия "
          f"про баг CRM не подходит. Проверьте эти строки вручную:")
    print(extra_unclear.to_string(index=False))

if len(lost_calls):
    print(f"\n4. НЕТ В GOOGLE SHEETS ({len(lost_calls)}) — есть в "
          f"Excel CRM, в Google Sheets этого звонка нет. "
          f"Кандидаты на дозапись в ячейке 6:")
    preview(lost_calls.drop(columns=["_d", "_row"]), "ПРОПУЩЕНО В GOOGLE SHEETS")

    # РАЗБОР ПРИЧИНЫ: первый ли это звонок своего дня.
    #
    # Дневные вкладки собираются из выгрузок на несколько дней, где
    # первый день буферный. Буфер спасает, ЕСЛИ CRM теряет одну запись на
    # весь период. Но если она теряет первую запись КАЖДОГО дня, то
    # рабочий день тоже лишается своего первого звонка — и буфер тут
    # бессилен. Отличить одно от другого можно только так: посмотреть,
    # стоят ли пропавшие звонки первыми в своём дне.
    first_of_day = {}
    for d_, g_ in month.groupby("_d"):
        first_of_day[d_] = g_.sort_values("_dt").iloc[0]["_key"]

    marks = []
    for _, r in lost_calls.iterrows():
        k = call_key(r["ID звонка"], r["Телефон"])
        marks.append(first_of_day.get(r["_d"]) == k)

    n_first = sum(marks)
    print(f"\n   Из них первых звонков своего дня: {n_first} из "
          f"{len(lost_calls)}")

    if n_first == len(lost_calls) and n_first > 1:
        print("   ВСЕ пропуски — первый звонок дня. Значит CRM теряет "
              "первую запись КАЖДОГО дня, а не только периода, и "
              "буферный день от этого не спасает: рабочий день тоже "
              "лишается своего первого звонка. Лечится дозаписью "
              "в ячейке 6 либо выгрузкой каждого дня отдельным файлом.")
    elif n_first:
        print(f"   Часть пропусков ({n_first}) — первый звонок дня, "
              f"остальные нет. Значит причин минимум две: к потере "
              f"первой записи добавилось что-то ещё, и остальные "
              f"строки стоит посмотреть вручную.")
    else:
        print("   Ни один не первый в своём дне — версия про потерю "
              "первой записи не подходит. Причина в другом: возможно, "
              "строка не доехала при записи в таблицу.")
else:
    print("\n4. В GOOGLE SHEETS ошибок не найдено — всё, что есть в "
          "Excel CRM, есть и там.")

if dupes:
    print("\nВНИМАНИЕ: внутри листа есть полностью повторяющиеся строки:")
    for d, n in dupes:
        print(f"   {d.strftime('%d.%m')}: {n} шт. — похоже на двойную вставку")

if bad_formula:
    print("\nВНИМАНИЕ: колонка G расходится с округлением вверх:")
    for d, n in bad_formula:
        print(f"   {d.strftime('%d.%m')}: строк {n}")
else:
    print("\nФормула округления в колонке G везде верна.")

## ЯЧЕЙКА 5 — СВЕРКА CRM / АТС ПО ИСХОДУ ЗВОНКА


In [ ]:
"""
Сверка исхода звонка: CRM против АТС.

В CRM попадают только звонки, которые реально дошли до оператора —
это установлено и проверено на данных в SQL-базе.
АТС видит всё, что пришло на линию, и у каждого вызова есть
результат: «Обработан первым оператором», «Потерян», «Обработан IVR».

Если оператор вместо того чтобы закрыть звонок как потерянный,
оставил его висеть как обычный входящий, — в CRM появится запись
о звонке, которого по данным АТС не было вообще. Эта ячейка
такие случаи находит.

СВЯЗКА CRM И АТС
Общего ID у систем нет, поэтому связываем по номеру телефона и
времени — тот же принцип и то же окно 180 секунд, что откалибровано
на июльских данных в SQL-базе: CRM всегда стартует ПОЗЖЕ
АТС на 9 секунд в среднем (сначала IVR и очередь, потом сам
разговор), максимум замеренного расхождения — 72 секунды. 180 —
запас поверх этого максимума.

ОТКУДА БЕРЁТСЯ АТС
Файлы приходят от супервайзера по одному дню, общего архива может
не быть. Ячейка сама сканирует папку и берёт все файлы, что похожи
на реестр АТС (по заголовку, не по имени файла) — сколько дней
найдётся, столько и сверится. Дата каждой строки читается из самого
файла, а не из его имени, поэтому неважно, как файлы называются.

ЧТО ЗНАЧИТ РЕЗУЛЬТАТ
Совпадение по телефону и времени в окне — сильный, но не железный
довод: если один номер звонил дважды за три минуты, окно теоретически
может зацепить не тот звонок. Поэтому ниже — «похоже на», а не
готовый вердикт, и кто из операторов на каком звонке стоял, показано
как факт из АТС, без выводов о том, кто виноват.
"""
import io

PBX_DIR = Path(BASE_DIR)     # запасной источник: файлы .xls
MATCH_WINDOW_SEC = 180             # то же окно, что откалибровано в SQL-базе


def read_pbx_from_sheets(year: int, mon: int) -> pd.DataFrame:
    """
    Читает реестр АТС прямо из дневных вкладок Google Sheets.

    Это основной источник: alpha_registry.ipynb пишет блок АТС под
    блоком CRM на каждой вкладке, с колонками «Дата | Время |
    Результат | Номер линии | Номер | ...». Значит данные за ВЕСЬ
    месяц уже лежат в таблице, и отдельные .xls-файлы за старые дни
    не нужны — их обычно и не сохраняют.

    Блок находим по шапке: ищем строку, где в первых трёх колонках
    стоят «Дата», «Время», «Результат». Отсчитывать фиксированное
    число строк нельзя — блок начинается после блока CRM, а тот
    каждый день разной высоты.
    """
    loader = SheetsLoader(CREDENTIALS, SPREADSHEET_ID)
    if USE_TITLE:
        loader.connect(title=SPREADSHEET_TITLE.format(
            month=RU_MONTHS[mon], year=year))
    else:
        loader.connect()

    d_from = date(year, mon, 1)
    d_to = date(year, mon, calendar.monthrange(year, mon)[1])
    sheets = loader.list_date_sheets(d_from, d_to)
    if not sheets:
        return pd.DataFrame()

    q = loader.quote_sheet
    ranges = [f"{q(t)}!A1:I3000" for t in sheets.values()]
    grids = []
    for i in range(0, len(ranges), 25):
        grids += loader.batch_get(ranges[i:i + 25])

    rows = []
    for d, grid in zip(sheets.keys(), grids):
        head = None
        for i, r in enumerate(grid or []):
            cells = [str(x).strip() for x in (r or [])[:3]]
            if cells[:3] == ["Дата", "Время", "Результат"]:
                head = i
                break
        if head is None:
            continue

        body = [r for r in (grid or [])[head + 1:]
                if r and len(r) > 2 and str(r[2]).strip()]
        if not body:
            continue

        # Колонку с номером АБОНЕНТА выбираем по данным, а не по
        # подписи. Причина: на старых вкладках подписи «Номер линии»
        # и «Номер» стояли наоборот, и чтение по позиции давало номер
        # линии — один и тот же во всех строках. Отличить их просто:
        # линия одна на весь день, абоненты разные. Берём ту из двух
        # колонок, где значений больше.
        def variety(idx):
            vals = {str(r[idx]).strip() for r in body
                    if len(r) > idx and str(r[idx]).strip()}
            return len(vals)

        phone_idx = 4 if variety(4) >= variety(3) else 3

        for r in body:
            rows.append({
                "call_date": d.strftime("%Y-%m-%d"),
                "call_time": str(r[1]).strip(),
                "result": str(r[2]).strip(),
                "phone": re.sub(r"\D", "",
                                str(r[phone_idx])
                                if len(r) > phone_idx else ""),
                "operator_name": str(r[6]).strip() if len(r) > 6 else "",
            })

    return pd.DataFrame(rows)


def _open_xls(path: Path):
    """Открывает .xls, не давая xlrd засорить вывод предупреждениями
    про размер сектора — на чтение это не влияет."""
    import xlrd
    return xlrd.open_workbook(path, logfile=io.StringIO())


def _to_sec(v) -> int:
    m = re.match(r"^(\d+):(\d{2}):(\d{2})$", str(v).strip())
    return int(m[1]) * 3600 + int(m[2]) * 60 + int(m[3]) if m else 0


def _norm_operator(name: str) -> str:
    m = re.fullmatch(r"\s*operator\s*(\d+)\s*", str(name), re.I)
    return f"Operator{int(m[1])}" if m else str(name).strip()


def read_pbx_registry(path: Path) -> pd.DataFrame:
    """
    Читает один файл реестра АТС (запасной путь).

    Нужен, если блока АТС на вкладках почему-то нет. Если
    заголовок не похож на реестр — возвращает пустой DataFrame и не
    падает, чтобы чужой файл в папке не остановил сверку.
    """
    try:
        ws = _open_xls(path).sheet_by_index(0)
    except Exception:
        return pd.DataFrame()

    hdr = [" ".join(str(ws.cell_value(1, c)).split()) for c in range(ws.ncols)]
    if hdr[:3] != ["Дата", "Время", "Результат"]:
        return pd.DataFrame()

    def txt(r, c):
        if c >= ws.ncols:
            return ""
        v = " ".join(str(ws.cell_value(r, c)).split())
        return "" if v == "+" else v

    rows = []
    for r in range(2, ws.nrows):
        d = txt(r, 0)
        m = re.fullmatch(r"(\d{2})\.(\d{2})\.(\d{4})", d)
        if not m:
            continue
        rows.append({
            "call_date": f"{m[3]}-{m[2]}-{m[1]}",
            "call_time": txt(r, 1),
            "result": txt(r, 2),
            "phone": re.sub(r"\D", "", txt(r, 4)),
            "operator_name": _norm_operator(txt(r, 7)),
        })
    return pd.DataFrame(rows)


def load_pbx_files(folder: Path) -> pd.DataFrame:
    """Собирает реестр из всех подходящих .xls в папке."""
    frames = [read_pbx_registry(p) for p in folder.glob("*.xls")]
    frames = [f for f in frames if len(f)]
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    return df.drop_duplicates(
        subset=["call_date", "call_time", "phone", "result"])


def find_outcome_mismatches(pbx: pd.DataFrame,
                            crm: pd.DataFrame) -> pd.DataFrame:
    """
    Сопоставляет звонки АТС и CRM и ищет расхождения в исходе.

    КАК СВЯЗЫВАЮТСЯ ЗВОНКИ
    Общего ID у систем нет. Раньше связывали просто: номер плюс окно
    180 секунд, и брали первый попавшийся звонок из окна. Это ломалось
    ровно там, где абонент звонил несколько раз подряд — обычное дело.

    Теперь так: звонки одного номера за день выстраиваются по времени
    в обеих системах и сопоставляются ПО ПОРЯДКУ — первый с первым,
    второй со вторым. Это и есть позиция в очереди. Каждый звонок CRM
    достаётся только одной строке АТС: занятый больше никому не
    предлагается.

    Окно остаётся, но как проверка, а не как способ выбора: CRM всегда
    стартует ПОЗЖЕ АТС (сперва IVR и очередь), в среднем на 9
    секунд, максимум замеренного — 72. Больше 180 — считаем, что это
    разные звонки.

    НАСКОЛЬКО МОЖНО ВЕРИТЬ
    Стопроцентной уверенности не даст ничего, пока у систем нет общего
    идентификатора. Но можно показать, на чём держится каждая пара,
    и это честнее одного ответа «совпало». Колонка «Надёжность»:

      точно        — кандидат был один, и оператор в обеих системах
                     совпал. Спорить почти не с чем.
      вероятно     — кандидат один, но сверить оператора не с чем
                     (в АТС он пуст — так бывает у потерянных).
      неоднозначно — в окне было несколько кандидатов. Пара выбрана
                     по порядку, и это разумно, но прежде чем
                     разговаривать с оператором — послушайте запись.
    """
    if pbx.empty or crm.empty:
        return pd.DataFrame()

    days = sorted(set(pbx["call_date"]))
    crm_in = crm[(crm["_d"].astype(str).isin(days)) &
                 (crm["Направление звонка"] == "Входящий")].copy()

    inf = pbx.copy()
    inf["_dt"] = pd.to_datetime(inf["call_date"] + " " + inf["call_time"],
                                errors="coerce")
    inf = inf[inf["_dt"].notna()].sort_values("_dt")

    used = set()          # индексы строк CRM, уже отданные в пару
    rows = []

    for (day, phone), grp in inf.groupby(["call_date", "phone"], sort=False):
        cand_all = crm_in[crm_in["_phone"] == phone].sort_values("_dt")

        for _, i in grp.sort_values("_dt").iterrows():
            lo, hi = i["_dt"], i["_dt"] + pd.Timedelta(seconds=MATCH_WINDOW_SEC)
            free = cand_all[(cand_all["_dt"] >= lo) & (cand_all["_dt"] <= hi) &
                            (~cand_all.index.isin(used))]

            match, conf = None, ""
            if len(free):
                # По порядку: берём самый ранний из свободных — он и
                # стоит в очереди на том же месте, что текущий АТС.
                match = free.iloc[0]
                used.add(free.index[0])

                op_inf = str(i.get("operator_name", "")).strip()
                op_crm = str(match.get("Оператор", "")).strip()
                if len(free) > 1:
                    conf = "неоднозначно"
                elif op_inf and op_crm:
                    conf = "точно" if op_inf == op_crm else "неоднозначно"
                else:
                    conf = "вероятно"

            res = i["result"]
            if res == "Потерян" and match is not None:
                cat = "АТС: потерян, в CRM есть как принятый"
            elif res == "Обработан первым оператором" and match is None:
                cat = "АТС: принят, в CRM звонка нет"
            elif res == "Обработан IVR" and match is not None:
                cat = "АТС: закрыт IVR, но в CRM есть как принятый"
            else:
                continue

            gap = (int((match["_dt"] - i["_dt"]).total_seconds())
                   if match is not None else None)
            rows.append({
                "Дата": day, "Время": i["call_time"], "Телефон": phone,
                "Категория": cat,
                "Разрыв, сек": gap,
                "Надёжность": conf,
                "Оператор (АТС)": i.get("operator_name", ""),
                "Оператор (CRM)": (match.get("Оператор", "")
                                   if match is not None else ""),
                "ID звонка в CRM": (int(match["ID звонка"])
                                    if match is not None
                                    and pd.notna(match.get("ID звонка"))
                                    else None),
            })

    return pd.DataFrame(rows)


if PROJECT != "alpha":
    print("Сверка с АТС умеет только Alpha.")
else:
    pbx = read_pbx_from_sheets(YEAR, MONTH)
    if pbx.empty:
        pbx = load_pbx_files(PBX_DIR)

    if pbx.empty:
        print("Реестр АТС не найден — сверка не выполнена.")
    else:
        mism = find_outcome_mismatches(pbx, month)

        if mism.empty:
            print(f"Проверено {len(pbx)} звонков АТС — "
                  f"ОШИБОК НЕТ.")
        else:
            print(f"Проверено {len(pbx)} звонков АТС — "
                  f"НАЙДЕНО ОШИБОК: {len(mism)}\n")
            for cat, g in mism.groupby("Категория"):
                print(f"{cat}: {len(g)}")
            # Показываем только надёжные пары: по ним можно работать
            # сразу. Спорные лежат в mism, но экран ими не забиваем.
            sure = mism[mism["Надёжность"] != "неоднозначно"]
            print(f"\nНадёжных {len(sure)}, спорных "
                  f"{len(mism) - len(sure)} (смотри mism)\n")
            preview(sure.drop(columns=["Категория", "Надёжность"]),
                    "ОШИБКИ")


## ЯЧЕЙКА 6 — ДОЗАПИСЬ ПРОПАВШИХ ЗВОНКОВ


In [ ]:
def write_back(dry_run: bool = True) -> None:
    """
    Дописывает потерянные звонки в конец нужных дневных вкладок.

    Колонки берутся из заголовка самой вкладки, а не задаются жёстко:
    у разных Google Sheets набор колонок отличается. Чего в Excel CRM нет,
    остаётся пустым.

    Колонка «Округление, минуты» пишется ФОРМУЛОЙ, а не числом —
    чтобы строка вела себя как остальные. Номер строки подставляется
    фактический, поэтому формулу собираем уже зная, куда пишем.

    Строки уходят в конец листа, а не на своё место по времени:
    вставка в середину сдвигает всё ниже и ломает формулы соседних
    ячеек. Отсортировать лист можно потом вручную.
    """
    def as_cell(v):
        """
        Готовит значение к записи в лист.

        Из Excel числа приходят float, и ID улетел бы как '6102454.0'.
        В следующей сверке такой номер уже не совпадёт с тем, что
        показывает лист, — и звонок снова будет числиться потерянным.
        """
        if v is None or (not isinstance(v, str) and pd.isna(v)):
            return ""
        if hasattr(v, "strftime"):
            return v.strftime("%Y-%m-%d %H:%M:%S")
        if isinstance(v, float) and float(v).is_integer():
            return int(v)
        return v

    if not len(lost_calls):
        print("Дописывать нечего — потерянных звонков нет.")
        return

    q = loader.quote_sheet
    heads = loader.batch_get(
        [f"{q(sheet_names[d])}!A1:Z1" for d in sorted(set(lost_calls['_d']))])

    payload, plan = [], []
    for (d, _), head in zip(
            sorted({(x, 1) for x in lost_calls["_d"]}), heads):
        cols = [str(c).strip() for c in (head[0] if head else [])]
        if not cols:
            print(f"  {d}: не удалось прочитать заголовок, пропускаю")
            continue

        part = lost_calls[lost_calls["_d"] == d]
        start = sheet_last_row[d] + 1
        values = []
        for n, (_, item) in enumerate(part.iterrows()):
            src = month.loc[item["_row"]]
            row_no = start + n
            row = []
            for c in cols:
                if c == "Округление, минуты":
                    row.append(minutes_formula(row_no))
                elif c in src.index:
                    row.append(as_cell(src[c]))
                else:
                    row.append("")
            values.append(row)

        rng = f"{q(sheet_names[d])}!A{start}"
        payload.append({"range": rng, "values": values})
        plan.append({"Вкладка": sheet_names[d], "Строк": len(values),
                     "С какой строки": start})

    print("ПЛАН ДОЗАПИСИ")
    print(pd.DataFrame(plan).to_string(index=False))

    if dry_run:
        if not ASK:
            print("\nDRY_RUN = True — ничего не записано.")
            print("Проверьте план выше, потом поставьте DRY_RUN = False.")
        return

    loader.sh.values_batch_update({
        "valueInputOption": "USER_ENTERED",
        "data": payload,
    })
    print(f"\nЗаписано вкладок: {len(payload)}, "
          f"строк: {sum(len(p['values']) for p in payload)}")
    print("Перепроверьте: перезапустите ячейки 3 и 4, "
          "разница должна стать нулевой.")


# Сначала план — план виден в любом случае, план_dry_run=True его
# не прячет. Дальше решаем, писать или нет.
write_back(dry_run=True)

if ASK:
    if len(lost_calls) and input(
            "\nЗаписать в Google Sheets? [нет/да]: ").strip().lower() \
            in ("да", "д", "yes", "y"):
        write_back(dry_run=False)
    elif len(lost_calls):
        print("Отменено — в Google Sheets ничего не записано.")
elif not DRY_RUN:
    write_back(dry_run=False)